# 02c · sensitivity runs: learning-rate schedule on the training loss
**Runtime: T4 GPU.** Run all. Same code, data, folds, seed (0) and settings as the main runs; the **only** change is that
ReduceLROnPlateau monitors the training loss instead of the validation loss (cell 3b). Writes to a NEW folder
`results/p2_cbis_runs_e200_lrtrain`; the main runs in `p2_cbis_runs_e200` are never touched.
30 runs (6 G/P models × 5 folds × seed 0), roughly 15–25 min each. Resumable: after a disconnect just Run all again.

**Decision rule (fixed before the runs):**
(1) The fix worked if no patch-wise run reaches lr <= 1e-6 by epoch 30. If it did not, the main results stand with the limitation stated.
(2) In-domain placement (resolved in the main 3-seed analysis): the claim "global better than patch-wise" holds if
patch − global on CBIS-DDSM is still negative for the circuit AND for Dense(4); otherwise it is reported as schedule-dependent.
(3) Circuit null result: holds if the circuit is not better than Dense(4) at either placement with a 95% CI excluding zero
(CBIS-DDSM and DMID), and the global circuit − Dense(4) difference on CBIS-DDSM stays within ±0.01 Dice.
(4) Bottleneck use: holds if the CBIS-DDSM const-drop is larger for global than for patch-wise placement for every block.
Contrasts that were not resolved in the main analysis (e.g. Q − Id) are reported but not used for the decision.


In [ ]:
# 1 · Drive, paths
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, importlib

PROJECT  = '/content/drive/MyDrive/quantum-placement'
DATA_DIR = f'{PROJECT}/data/processed/cbis_mass_256'
CODE_DIR = f'{PROJECT}/code'
RUNS_DIR = f'{PROJECT}/results/p2_cbis_runs_e200_lrtrain'   # sensitivity runs (new folder; main runs are never touched)
MAIN_DIR = f'{PROJECT}/results/p2_cbis_runs_e200'           # main runs, read only
TIME_DIR = f'{PROJECT}/results/_timing'
QSEG_DIR = '/content/drive/MyDrive/quantum-placement/code'            # folder that contains the qseg/ package
if not os.path.exists(f'{QSEG_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True)
    assert hits, 'qseg package not found on Drive'
    QSEG_DIR = os.path.dirname(os.path.dirname(hits[0]))
for d in (CODE_DIR, RUNS_DIR, TIME_DIR): os.makedirs(d, exist_ok=True)
assert os.path.exists(f'{DATA_DIR}/folds.csv'), 'Run notebook 01 first'
for p in (QSEG_DIR, CODE_DIR):
    if p not in sys.path: sys.path.insert(0, p)
print('qseg :', QSEG_DIR); print('data :', DATA_DIR); print('runs :', RUNS_DIR)


In [ ]:
# 2 · Use the model library already on Drive (written by notebook 02; not rewritten here)
import importlib, qseg2; importlib.reload(qseg2)
import tensorflow as tf, numpy as np, pandas as pd
print('qseg2 from:', qseg2.__file__)
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'No GPU: Runtime -> Change runtime type -> T4 GPU'


In [ ]:
# 3 · Data + shared config (base settings of the qseg package)
from qseg.config import Config
cfg = Config(work_dir=RUNS_DIR, max_epochs=200)   # v3: up to 200 epochs; batch 16, Adam 1e-4, early stop patience 10 after epoch 15
X, Y, DF = qseg2.load_cbis(DATA_DIR)
print(X.shape, Y.shape, 'ROIs:', len(DF), '| patients:', DF.patient.nunique())
print(DF.groupby('fold').agg(rois=('roi_id','size'), patients=('patient','nunique')).T.to_string())


In [ ]:
# 3b · The ONLY change vs the main runs: the learning-rate schedule watches the TRAINING loss.
# In the main runs ReduceLROnPlateau watched val_loss; an early validation-loss spike (BatchNorm moving
# statistics still settling) cut the learning rate every 4 epochs, so patch-wise models and 'full' reached
# lr <= 1e-6 by epoch 30. Factor 0.5, patience 4, min_lr 1e-7 stay the same. Early stopping and the best
# checkpoint still use val_loss (start after epoch 15, patience 10), exactly as before.
_Plateau = tf.keras.callbacks.ReduceLROnPlateau
if not getattr(_Plateau, '_p2_train_loss', False):
    class TrainLossPlateau(_Plateau):
        _p2_train_loss = True
        def __init__(self, monitor='val_loss', **kw):
            super().__init__(monitor='loss', **kw)
    tf.keras.callbacks.ReduceLROnPlateau = TrainLossPlateau
assert qseg2.tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4).monitor == 'loss'
print('LR schedule now monitors:', qseg2.tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4).monitor)


In [ ]:
# 4 · Training: seed 0, six G/P models, 5 folds = 30 runs (resumable). Order: fold by fold, all six models.
MODELS_TO_RUN = ['g_q', 'g_cl', 'g_id', 'p_q', 'p_cl', 'p_id']
FOLDS, SEEDS  = [0, 1, 2, 3, 4], [0]
DATA = (X, Y, DF)
N = len(SEEDS) * len(FOLDS) * len(MODELS_TO_RUN)
def n_done():
    return sum(os.path.exists(f'{RUNS_DIR}/{m}_fold{f}_seed{s}/summary.json') for s in SEEDS for f in FOLDS for m in MODELS_TO_RUN)
def lr_check(run):
    h = pd.read_csv(f'{RUNS_DIR}/{run}/history.csv')
    col = 'learning_rate' if 'learning_rate' in h else 'lr'
    low = h.index[h[col] <= 1e-6]
    return (int(low[0]) + 1) if len(low) else None, len(h)
import time
print(f'already finished: {n_done()} / {N}')
for seed in SEEDS:
    for fold in FOLDS:
        for name in MODELS_TO_RUN:
            run = f'{name}_fold{fold}_seed{seed}'
            qseg2.run_fold(cfg, DATA, name, fold, seed, out_root=RUNS_DIR, min_epochs=15, verbose=0)
            ep_low, n_ep = lr_check(run)
            print(time.strftime('%H:%M UTC', time.gmtime()), f'| {run}: {n_ep} epochs, lr<=1e-6 from epoch {ep_low} | done {n_done()} / {N}', flush=True)
print('all requested runs finished')


In [ ]:
# 5 · Auto-shutdown: save Drive, then release the GPU (runs only after cell 4 finishes)
import time
from google.colab import drive, runtime
print('training cell finished at', time.strftime('%H:%M:%S UTC', time.gmtime()))
drive.flush_and_unmount()
print('Drive flushed')
time.sleep(30)
runtime.unassign()
